In [ ]:
# Auto reload
%load_ext autoreload
%autoreload 2

# library imports
from matplotlib import pyplot as plt
import numpy as np

# import PGL Eyelink interface
from pgl import pglEyelink, pglEyelinkData, pglTask, pglParameter

# Load PGL libraries and start a PGL window
from pgl import pgl, pglExperiment
pgl = pgl()
e = pglExperiment(pgl, settingsName="ViewPixx", experimentName="Eyelink test")

# close any existing windows
pgl.cleanUp()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:35:20) [Clang 16.0.6 ]
(pgl:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.4.1
(pgl:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl:checkOS)   Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pglBase) Main library instance created
(

In [ ]:

e.initScreen()
eyelink = pglEyelink(pgl)

In [ ]:
# TODO
# 1 abort calibration key
# *** 2 Eat all keys
# *** 3 setting for eccentricity of calibration
# 4 Check Read / Write EDF
# 5 Send triggerq to EDF 
# 6 Handle more gracefully stopping keyboard - with interrupt handler for eyelink
# 7 Force a
# qQccept of point
# 8 Display crosses on eye screen
# 9 
#eyelink.setCustomCalibrationPoints(margin=0.7, numPoints=9)
#eyelink.calibrate()
#e.endScreen()
eyelink.openEDF('test')
eyelink.start()
pgl.waitSecs(2)
eyelink.sendMessage("Test 1")
pgl.waitSecs(2)
eyelink.sendMessage("Test 2")
pgl.waitSecs(1)
eyelink.stop()
x = eyelink.getEDF('test','/Users/justin/data')


In [ ]:
d=pglEyelinkData('/Users/justin/Desktop/test.asc')
print(d.trials[0]['saccades'])
d.displayTrials()
#d=pglEyelinkData('/Users/justin/data/test.asc')

In [4]:
e = pglExperiment(experimentName="pglFixTask")

(pglExperiment) No settings provided, using default settings.
(pglExperiment:load) Data directory: /Users/justin/data/pglFixTask/s0000
Experiment name: pglFixTask | Subject ID: s0000
1. Wednesday May 13, 2026 1:25PM | nVols: 0 | 23s | Default experiment (20260513_132527)
2. Wednesday May 13, 2026 4:04PM | nVols: 0 | 36s | Default experiment (20260513_160443)
3. Wednesday May 13, 2026 4:09PM | nVols: 0 | 1 min 8s | Default experiment (20260513_160928)
4. Wednesday May 13, 2026 5:15PM | nVols: 0 | 1 min 8s | Default experiment (20260513_171531)
5. Wednesday May 13, 2026 5:19PM | nVols: 0 | 1 min 11s | Default experiment (20260513_171931)
(pglExperiment:load) Loading experimentdata from: /Users/justin/data/pglFixTask/s0000/20260513_171931
(pglExperiment:load) Loading task data from: /Users/justin/data/pglFixTask/s0000/20260513_171931/eyeFixationTask


In [16]:
e.print()

Experiment: Default experiment | Subject ID: s0000
Duration: 1 min 11s
Display: VIEWPixx3D: Unknown 1920x1080 @ 60Hz 80.36x48.18 deg 52.20x29.50 cm at 34.00 cm 
Number of volume triggers: 0
Task: Eye Fixation Task | Trials: 48
duration=71.50s | startTime=1152956.33 | endTime=1153027.83
Trial 1 at 0.00s: fixIndex=3
Trial 2 at 1.50s: fixIndex=6
Trial 3 at 3.02s: fixIndex=2
Trial 4 at 4.53s: fixIndex=5
Trial 5 at 6.03s: fixIndex=4
Trial 6 at 7.53s: fixIndex=0
Trial 7 at 9.05s: fixIndex=1
Trial 8 at 10.55s: fixIndex=7
Trial 9 at 12.07s: fixIndex=8
Trial 10 at 13.58s: fixIndex=4
Trial 11 at 15.10s: fixIndex=7
Trial 12 at 16.60s: fixIndex=3
Trial 13 at 18.12s: fixIndex=2
Trial 14 at 19.63s: fixIndex=1
Trial 15 at 21.15s: fixIndex=5
Trial 16 at 22.67s: fixIndex=6
Trial 17 at 24.17s: fixIndex=0
Trial 18 at 25.67s: fixIndex=8
Trial 19 at 27.18s: fixIndex=7
Trial 20 at 28.70s: fixIndex=4
Trial 21 at 30.22s: fixIndex=1
Trial 22 at 31.73s: fixIndex=3
Trial 23 at 33.25s: fixIndex=2
Trial 24 at 34.7

In [ ]:
#e.display()
#e.tasks[0].data
#e.print()

print(e.tasks[0].data.events[6].timestamp)
#trialEvent.trialNum

In [ ]:
print(d)
#d.parseMessages()
#plt.plot(d.samples['time'],d.samples['x'],'.')
#d.samples
#d.messages
#d.fixations
#d.saccades
d.blinks

In [ ]:
# Set up eye fixation task
class pgFixationTask(pglTask):
    
    ########################
    def __init__(self, pgl, eyelink):
        super().__init__(pgl)
        self.eyelink = eyelink
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Eye Fixation Task"
        self.settings.seglen = [1.5]
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            'nPoints':9,
            'width':10,
            'height':10
        }        
        p = self.settings.fixedParameters

        # compute positions of calibration points
        if p['nPoints']==9:
            x = p['width']*0.5
            y = p['height']*0.5
            self.settings.fixedParameters['calibrationPoints'] = [(0,0),(-x,y),(0,y),(x,y),(-x,0),(x,0),(-x,-y),(0,-y),(x,-y)]
        else:
            raise ValueError(f"Unsupported number of points: {p['nPoints']}")
        
        fixIndex = pglParameter('fixIndex',np.arange(p['nPoints']))
        self.addParameter(fixIndex)
        
    ########################
    def updateScreen(self):
        if self.state.currentSegment==0:
            # get the current fix Index
            fixIndex = self.currentParams['fixIndex']
            # get the position
            x = self.settings.fixedParameters['calibrationPoints'][fixIndex][0]
            y = self.settings.fixedParameters['calibrationPoints'][fixIndex][1]
            self.pgl.circle(0.25,x,y,fill=True)
            self.pgl.fixationCross(0.25,x,y,color=[1,0,0])
    def startTrial(self, startTime):
        # call super
        super().startTrial(startTime)
        # send a message to eyelink
        eyelink.sendMessage(f"pgl: trialStart trialNum={self.state.currentTrial}  timestamp={startTime}")
    def startSegment(self, updateTime):
        # call super
        super().startSegment(updateTime)
        # send a message to eyelink
        eyelink.sendMessage(f"pgl: segmentStart segmentNum={self.state.currentSegment} trialNum={self.state.currentTrial} timestamp={updateTime}")

# initialize task
fixTask = pgFixationTask(pgl,eyelink)


In [ ]:
e.initScreen()
e.addTask(fixTask)

eyelink.setCustomCalibrationPoints(margin=0.7, numPoints=9)
eyelink.calibrate()

eyelink.openEDF('test')
eyelink.start()
e.run()
eyelink.stop()
eyelink.getEDF('test','/Users/justin/data')

e.endScreen()
e.pgl.close()


In [ ]:
eyelink.openEDF('test')
eyelink.start()
pgl.waitSecs(2)
eyelink.sendMessage("Test 1")
pgl.waitSecs(2)
eyelink.sendMessage("Test 2")
pgl.waitSecs(1)
eyelink.stop()
x = eyelink.getEDF('test','/Users/justin/data')

In [ ]:
help(pgl.circle)

In [ ]:
#from pgl import pglEyelink
pgl.cleanUp()
pgl.open(0,800,600)
pgl.visualAngle(57,30,20)
from pgl import pglEyelinkCustomDisplay
#eyelink = pglEyelink(pgl)
#customDisplay = pglEyelinkCustomDisplay(pgl,eyelink)

customDisplay = pglEyelinkCustomDisplay(pgl)

customDisplay.clear_cal_display()
customDisplay.draw_line(00,0,500,500,0)
customDisplay.draw_lozenge(100,200,100,300,0)
customDisplay.draw_cal_target(300,300)
#customDisplay.get_input_key()
#pgl.flush()

In [ ]:
eyelink.close()

In [ ]:
import pylink

def setup_eye_link(eyelink_address="100.1.1.1"):
    try:
        # Create an EyeLink tracker object.
        el_tracker = pylink.EyeLink(eyelink_address)
        print(f"Connecting to EyeLink at {eyelink_address}...")
        el_tracker.open()  # Open the connection to the EyeLink
        print("EyeLink connected.")

        # Check connection status
        if el_tracker.isConnected():
            print("EyeLink is connected.")
        else:
            print("EyeLink is not connected.")
            return

        # Setup and calibrate the tracker
        print("Performing tracker setup...")
        el_tracker.doTrackerSetup()  # Setup the tracker (calibration and validation)
        print("Tracker setup completed.")

        # Optionally, you could start recording data here
        # el_tracker.startRecording(1, 1, 1, 1)  # Start recording

        # Close the connection when finished
        el_tracker.close()
        print("EyeLink connection closed.")

    except pylink.EyeLinkError as e:
        print(f"An EyeLink error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Run the setup
setup_eye_link()